# 英文教材批次處理（語音辨識 + 翻譯）

在 Colab 的 GPU 上，把 YouTube 影片或 Podcast 集數批次轉成逐字稿、分句、翻譯，結果存到 Google Drive；
完成後下載回本機，用 `english-app/backend/scripts/import_batch.py` 匯入到主系統的 `app.sqlite`。

**這是獨立於主系統的批次工具**：主系統平常怎麼處理影片（`services/pipeline.py`）完全不受影響，
只有分句規則（`services/segmenter.py`）、幻聽過濾（`services/hallucination.py`）、
長音檔分段（`services/chunked.py`）、翻譯提示詞（`services/prompts.py`）這幾個純 Python、
跟資料庫和 FastAPI 無關的模組，會直接從你的 repo clone 下來重用，確保跟本機處理的規則一致。

## 使用前

1. **選 GPU**：選單「執行階段 → 變更執行階段類型」→ 硬體加速器選 GPU（T4 免費版就夠用）。
2. **翻譯改用 Colab 自己的 GPU 跑 Ollama + qwen3:8b**，不呼叫任何雲端 API、不用金鑰——筆記本會自動安裝
   Ollama、下載 qwen3:8b。第一次下載模型跟 Whisper 模型都要花幾分鐘，之後同一個 session 內是快取的。
3. 如果你的 repo（`horaceweng/my_saylo`）是私人的，`git clone` 那一格會失敗，需要改用個人存取權杖，
   見該格下面的說明。

## 已知限制（跑之前要知道）

- **免費 GPU 有時數上限、會被閒置回收**：處理結果每完成一份就立刻寫進 Google Drive，重新執行這份筆記本會自動跳過已完成的項目，所以斷線後接著跑就好，不會重來。
- **大量下載 YouTube 音訊可能被暫時限流**：如果中途一直下載失敗，先間隔一段時間再繼續，或減少 `MAX_ITEMS_PER_LEVEL`。
- 逐字時間戳的精準度取決於你選的 Whisper 模型大小。這份筆記本預設用 `large-v3-turbo`——跟主系統本機用的
  Whisper 模型是同一個，準確度夠、又比 `large-v3` 快很多，VRAM 用量也小很多，讓同一張 GPU 還能同時跑 Ollama 的
  qwen3:8b。免費 T4 有 16 GB VRAM，qwen3:8b（約 5 GB）+ `large-v3-turbo`（約 1.5 GB）同時載入完全沒問題；
  如果你想換更準的 `large-v3`（約 3 GB），也還在 T4 的容量內，只是轉字幕會慢不少。
- **Ollama 跑在 Colab 這台虛擬機裡，關掉筆記本或斷線，Ollama 和已下載的模型都會消失**：重新執行時筆記本會
  重新安裝 Ollama、重新下載 qwen3:8b（模型下載通常比重新處理已完成的項目快，因為那些項目本來就會被跳過）。
- 這份筆記本沒有在真正的 Colab 環境跑過（我這邊沒有 GPU 也沒有你的 API 金鑰），語音辨識與翻譯呼叫本身**沒有實際驗證過**；
  分句、幻聽過濾、批次翻譯重試這幾塊純邏輯，已經用假資料在本機測試過，程式對不對可以放心，
  但整體跑起來順不順、辨識準不準，還是要你自己跑第一批（建議先設 `MAX_ITEMS_PER_LEVEL = 2` 試跑）才知道。


## 1. 檢查 GPU、安裝套件

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print("⚠️ 沒有偵測到 GPU，請先在執行階段設定改用 GPU")


In [ ]:
!pip install -q faster-whisper yt-dlp httpx pydantic pydantic-settings


## 2. 掛載 Google Drive（結果存這裡，斷線也不會丟）

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
OUTPUT_ROOT = Path("/content/drive/MyDrive/english_lab_batch")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("結果會存到：", OUTPUT_ROOT)


## 3. 取得分句／幻聽過濾／翻譯提示詞的程式碼

直接 clone 你的 repo，重用 `english-app/backend/app/services/` 裡幾個跟資料庫、FastAPI 無關的純 Python 模組，
確保跟主系統本機處理影片時用的是同一套規則。**如果 repo 是私人的**，把下面這行的網址換成
`https://<你的-GitHub-token>@github.com/horaceweng/my_saylo.git`（GitHub → Settings → Developer settings →
Personal access tokens 產生一個，只需要 repo 的讀取權限）。


In [ ]:
REPO_URL = "https://github.com/horaceweng/my_saylo.git"  # 私人 repo 請改成上面說的含 token 網址

!rm -rf /content/my_saylo
!git clone --depth 1 "{REPO_URL}" /content/my_saylo

import sys
sys.path.insert(0, "/content/my_saylo/backend")

from app.services.segmenter import Word, split_sentences
from app.services.hallucination import RawSegment, drop_hallucinations
from app.services.chunked import transcribe_in_pieces, CHUNK_SECONDS, OVERLAP_SECONDS
from app.services.prompts import TRANSLATE_SYSTEM, TranslationBatch, translate_user_prompt

print("已載入分句、幻聽過濾、分段、翻譯提示詞模組（跟主系統同一份程式碼）")


## 4. 設定：要跑哪些等級、哪些種類、跑幾份、用哪個 Whisper 模型

`materials.csv` 是 `english-app/scripts/materials/materials.csv`（clone 下來的 repo 裡就有），
欄位是 `level, type, source, title, url, duration_sec, published`。


In [ ]:
import csv

MATERIALS_CSV = "/content/my_saylo/scripts/materials/materials.csv"

# 先用小數字試跑一次，順利了再調大或設成 None（全部跑）
LEVELS = ["A2", "B1", "B2", "C1+"]          # 要跑的等級
TYPES = ["video", "podcast"]                # 要跑的種類
MAX_ITEMS_PER_LEVEL_TYPE = 2                # 每個「等級 x 種類」最多跑幾份；先小規模試跑，改成 None 表示全部

WHISPER_MODEL_SIZE = "large-v3-turbo"       # tiny/base/small/medium/large-v3/large-v3-turbo；跟主系統本機同一個模型，準確又比 large-v3 快、省 VRAM
WHISPER_COMPUTE_TYPE = "float16"            # GPU 用 float16；用 CPU 的話改成 "int8"

with open(MATERIALS_CSV, encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))

def selected_rows():
    picked = []
    for level in LEVELS:
        for type_ in TYPES:
            rows = [r for r in all_rows if r["level"] == level and r["type"] == type_]
            picked += rows if MAX_ITEMS_PER_LEVEL_TYPE is None else rows[:MAX_ITEMS_PER_LEVEL_TYPE]
    return picked

rows = selected_rows()
print(f"這次會處理 {len(rows)} 份（{MATERIALS_CSV} 裡總共有 {len(all_rows)} 份）")


## 5. 安裝 Ollama，下載 qwen3:8b（翻譯用）

翻譯直接在 Colab 這台機器上用 Ollama 跑，跟主系統平常用 Ollama 翻譯的方式完全一樣（重用同一份
`OllamaProvider`），不呼叫任何雲端 API、不需要金鑰。qwen3:8b 大約 5 GB，第一次下載要幾分鐘。


In [ ]:
!sudo apt-get install -y pciutils zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama || (echo "❌ ollama 沒有安裝成功，往上找 curl 那一步的錯誤訊息" && exit 1)

import subprocess
import time

import httpx

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

OLLAMA_BASE_URL = "http://localhost:11434"
TRANSLATE_MODEL = "qwen3:8b"

for _ in range(30):
    try:
        httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        break
    except httpx.HTTPError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama 伺服器沒有在時間內啟動")

!ollama pull {TRANSLATE_MODEL}
print("Ollama 就緒，翻譯模型：", TRANSLATE_MODEL)


## 6. 下載音訊：YouTube 用 yt-dlp，Podcast 直接下載音檔網址
**如果 YouTube 下載一直失敗，出現「Sign in to confirm you're not a bot」**：Colab 的 IP 常被 YouTube
的反機器人機制擋下，比家用網路常見很多。解法是帶一份從瀏覽器匯出的 cookies：

1. 用一個**備用帳號**（不建議用主帳號）登入 youtube.com，用瀏覽器擴充功能（例如「Get cookies.txt LOCALLY」）
   匯出 cookies，存成 `cookies.txt`。
2. Colab 左側「檔案」面板上傳這個檔案，確定路徑是 `/content/cookies.txt`。
3. 下面這格會自動偵測這個檔案存在就使用；不存在的話還是照樣跑（可能部分影片會遇到同樣的錯誤）。

這份 cookies 等於你帳號當下的登入狀態，不是密碼，但有效期內能冒充該帳號操作 YouTube——**不要存進
Google Drive**，`/content/` 底下的檔案關掉 Colab session 就會消失。


In [ ]:
import subprocess
import httpx
from pathlib import Path

AUDIO_DIR = Path("/content/audio")
AUDIO_DIR.mkdir(exist_ok=True)


# Colab's IP range gets YouTube's "Sign in to confirm you're not a bot" far more often than a home
# network — confirmed while building this notebook. yt-dlp's own fix is a cookies file exported from a
# real, logged-in browser session (see the markdown above this cell for how to make and upload one).
# It is optional: without it, some videos work and others hit the bot check.
COOKIES_FILE = Path("/content/cookies.txt")


def download_video_audio(url: str, dest: Path) -> None:
    """yt-dlp → 16 kHz mono wav, so faster-whisper reads it directly without an extra convert step."""
    cmd = ["yt-dlp", "--no-warnings", "--extractor-args", "youtube:player_client=android",
           "-x", "--audio-format", "wav", "--postprocessor-args", "ffmpeg:-ar 16000 -ac 1",
           "-o", str(dest.with_suffix("")) + ".%(ext)s"]
    if COOKIES_FILE.exists():
        cmd += ["--cookies", str(COOKIES_FILE)]
    cmd.append(url)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"yt-dlp 失敗：{(result.stderr or result.stdout).strip()[-600:]}")


# Some hosts (confirmed: VOA's audio CDN) return 403 for httpx's own User-Agent ("python-httpx/…") while
# a browser-looking one works fine — not an IP block, just a naive bot filter.
BROWSER_USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"


def download_podcast_audio(url: str, dest: Path) -> None:
    """A podcast's audio is already a direct file link: fetch it and let ffmpeg (inside faster-whisper's
    decode_audio) handle whatever format it is in."""
    with httpx.stream("GET", url, follow_redirects=True, timeout=120, headers={"User-Agent": BROWSER_USER_AGENT}) as resp:
        resp.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in resp.iter_bytes(1 << 20):
                f.write(chunk)


def download_audio(row: dict, dest: Path) -> None:
    if row["type"] == "video":
        download_video_audio(row["url"], dest)
    else:
        download_podcast_audio(row["url"], dest)


## 7. Whisper：轉逐字稿，套用跟主系統一樣的分句與幻聽過濾規則

In [ ]:
from dataclasses import dataclass
from faster_whisper import WhisperModel
from faster_whisper.audio import decode_audio

print("載入 Whisper 模型（第一次要下載，之後同一個 session 內是快取的）…")
whisper_model = WhisperModel(WHISPER_MODEL_SIZE, device="cuda", compute_type=WHISPER_COMPUTE_TYPE)


def raw_segments_from_faster_whisper(fw_segments, offset: float = 0.0) -> list[RawSegment]:
    """faster-whisper's Segment objects → the app's RawSegment, so the existing hallucination filter
    and sentence splitter (segmenter.py) can be reused exactly as they are on the live app."""
    out = []
    for seg in fw_segments:
        words = [Word(w.word, w.start + offset, w.end + offset, w.probability) for w in (seg.words or [])]
        out.append(RawSegment(
            start=seg.start + offset, end=seg.end + offset, text=seg.text,
            avg_logprob=seg.avg_logprob, no_speech_prob=seg.no_speech_prob,
            compression_ratio=seg.compression_ratio, words=words,
        ))
    return out


def make_transcribe_span(audio):
    """Returns transcribe_span(t0, t1) -> (words, doubtful words), the shape transcribe_in_pieces expects
    (see services/chunked.py and services/pipeline.py._run on the live app)."""
    sample_rate = 16000

    def transcribe_span(t0: float, t1: float):
        piece = audio[int(t0 * sample_rate):int(t1 * sample_rate)]
        segments, _info = whisper_model.transcribe(piece, word_timestamps=True, language="en", condition_on_previous_text=False)
        raw = raw_segments_from_faster_whisper(list(segments), offset=t0)
        kept, dropped = drop_hallucinations(raw, start_at=t0, keep_all_if_doubtful=False)
        return [w for seg in kept for w in seg.words], [w for d in dropped for w in d.segment.words]

    return transcribe_span


def transcribe_file(path: Path):
    """The whole file → a flat list of Sentence (see segmenter.py), same approach as pipeline.py._run:
    cut into CHUNK_SECONDS pieces with OVERLAP_SECONDS, so a 60-minute file does not need 60 minutes of
    context in memory at once."""
    audio = decode_audio(str(path), sampling_rate=16000)
    duration = len(audio) / 16000
    doubtful_all = []
    sentences = []
    for piece in transcribe_in_pieces(make_transcribe_span(audio), duration, chunk=CHUNK_SECONDS, overlap=OVERLAP_SECONDS):
        sentences += piece.sentences
        doubtful_all += piece.doubtful
    if not sentences and doubtful_all:
        # nothing looked reliable anywhere (poor audio): show it rather than silently produce an empty video
        sentences = split_sentences(doubtful_all)
    return sentences, duration


## 8. 翻譯：跟主系統一樣的提示詞，缺漏的句子單獨重問一次

In [ ]:
from app.services.llm import OllamaProvider, chat_json

translate_provider = OllamaProvider(base_url=OLLAMA_BASE_URL, model=TRANSLATE_MODEL)


async def translate_sentences(sentences: list[str], batch_size: int = 20) -> list[str]:
    """Same approach as services/pipeline.py.translate_sentences: batch first (Ollama processes requests
    serially, so this is about staying consistent with the sentences before/after, not about speed), then
    retry only the sentences the model skipped, one at a time."""
    async def ask(batch: list[str]) -> dict[str, str]:
        try:
            result = await chat_json(translate_provider, TRANSLATE_SYSTEM, translate_user_prompt(batch), TranslationBatch)
        except Exception:  # noqa: BLE001 - a bad reply from this batch just means the sentence gets a solo retry
            return {}
        return {k.strip(): v.strip() for k, v in result.translations.items() if v.strip()}

    out: list[str] = []
    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]
        got = await ask(batch)
        for i, sentence in enumerate(batch):
            text = got.get(str(i + 1))
            if not text:
                text = (await ask([sentence])).get("1", "")
            out.append(text)
    return out


## 9. 批次處理：每處理完一份就存進 Drive，已完成的自動跳過

輸出格式跟 `backend/app/services/batch_import.py` 的 `parse_item` 要求的欄位一致，
本機執行 `scripts/import_batch.py` 時可以直接吃這些 JSON。


In [ ]:
import hashlib
import json
import time
import traceback

FAILURES_LOG = OUTPUT_ROOT / "failures.csv"


def output_path(row: dict) -> Path:
    safe_id = hashlib.sha1(row["url"].encode()).hexdigest()[:16]
    folder = OUTPUT_ROOT / row["level"] / row["type"]
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{safe_id}.json"


def log_failure(row: dict, error: Exception) -> None:
    is_new = not FAILURES_LOG.exists()
    with open(FAILURES_LOG, "a", encoding="utf-8") as f:
        if is_new:
            f.write("level,type,url,title,error\n")
        message = str(error).replace(",", ";").replace("\n", " ")[:200]
        f.write(f'{row["level"]},{row["type"]},{row["url"]},"{row["title"][:60]}","{message}"\n')


async def process_row(row: dict) -> None:
    out_path = output_path(row)
    if out_path.exists():
        print(f"− 已經做過，略過：{row['title'][:60]}")
        return

    audio_path = AUDIO_DIR / "current.wav"
    audio_path.unlink(missing_ok=True)
    download_audio(row, audio_path)

    sentences, duration = transcribe_file(audio_path)
    if not sentences:
        raise RuntimeError("沒有辨識到任何英文語音")
    translations = await translate_sentences([s.text for s in sentences])

    result = {
        "kind": row["type"], "source_url": row["url"], "title": row["title"], "duration": duration,
        "segments": [
            {"text": s.text, "start": s.start, "end": s.end, "translation": tr,
             "words": [{"text": w.text, "start": w.start, "end": w.end, "prob": w.prob} for w in s.words]}
            for s, tr in zip(sentences, translations)
        ],
    }
    out_path.write_text(json.dumps(result, ensure_ascii=False), encoding="utf-8")
    print(f"✓ {row['level']}/{row['type']} 「{row['title'][:50]}」：{len(sentences)} 句")


async def run_batch(rows: list[dict]) -> None:
    for i, row in enumerate(rows, 1):
        t0 = time.monotonic()
        try:
            await process_row(row)
        except Exception as e:  # noqa: BLE001 - one bad item must not stop the whole batch
            print(f"✗ {row['title'][:50]}：{e}")
            traceback.print_exc(limit=1)
            log_failure(row, e)
        print(f"  ({i}/{len(rows)}，{time.monotonic() - t0:.0f}s)")


await run_batch(rows)
print("\n全部跑完。失敗清單（如果有）在：", FAILURES_LOG)


## 10. 完成後：把結果帶回本機

Google Drive 裡的 `english_lab_batch/` 資料夾就是全部結果（斷線重跑也不會重複，已完成的會自動跳過）。
下面這格把它打包成一個 zip，從 Colab 左側的檔案面板下載；也可以直接在本機把整個 Google Drive 資料夾同步下來。

下載回本機後，在 `english-app/backend` 執行：

```
uv run python scripts/import_batch.py /path/to/english_lab_batch
```

會把每一份寫進 `data/app.sqlite`，跟主系統本機處理出來的影片一樣可以直接在畫面上打開；已經匯入過的（同一個
影片或集數的網址）會自動跳過，不會重複。


In [ ]:
import shutil

zip_path = "/content/english_lab_batch"
shutil.make_archive(zip_path, "zip", OUTPUT_ROOT)
print("打包完成：", zip_path + ".zip", "（在左側檔案面板下載）")
